<a href="https://colab.research.google.com/github/yeseul-04/kfa-ai-agent/blob/main/w03_%EC%97%94%EC%A7%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 3회차 엔진 v0: DART 한 종목을 출처 번호까지

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phm0305/KFA-ai-agent-study/blob/main/notebooks/03_engine_v0.ipynb)

2회차에 만든 공통 표(15칸)에 **줄 이름표(rid)**와 **출처 번호([S#])**를 붙여요.
이 둘이 있어야 나중에 글에 나온 숫자를 공통 표의 줄과 원문 공시까지 거꾸로 따라갈 수 있어요.

```
DART 2개 연도 ─────┐
출처카드 (수동 입구) ─┼──▶ 공통 표 15칸 ──▶ 꼬리표 점검 ──▶ rid · [S#] 붙이기 ──▶ CSV 저장
내 소스 × 3종목 ────┘
```

**사용법:** 아래 설정 셀의 `STOCK_CODE`, `YEAR` 두 줄만 바꾸고 위 메뉴 **런타임 → 모두 실행**.
"보안 비밀 액세스 허용" 창이 뜨면 **허용**을 눌러요. 중간에 뭐가 실패해도 멈추지 않고 끝까지 가요.

**설정**

- `STOCK_CODE`: 종목코드 6자리. 따옴표 안에, 앞의 0까지 적어요
- `YEAR`: 사업보고서 연도. **이 해와 그 전년도** 두 해를 받아요. 아직 사업보고서가 안 나온 해면 DART가 '데이터 없음'을 줘요
- `USE_DRIVE`: 2회차 `KFA_자료` 폴더와 출처카드를 쓸 때만 `True` (첫 글자 대문자)
- `CARD_SHEET`: 출처카드 구글 시트 제목

In [43]:
STOCK_CODE = "011170"
YEAR = 2025
USE_DRIVE = True
CARD_SHEET = "출처카드"

In [44]:
#@title 📦 설치 (1분) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "finance-datareader", "python-dotenv"],
                      capture_output=True, text=True)
print("✅ 설치 완료" if proc.returncode == 0 else "❌ 설치 실패: 런타임 → 런타임 다시 시작 → 모두 실행")

✅ 설치 완료


In [45]:
#@title 🧰 공통 도구 (고치지 말고 실행만. 코드가 궁금하면 이 셀을 더블클릭) { display-mode: "form" }
"""KFA AI agent study: shared collection toolkit (2nd session template).

Every source, whatever its collection level (1 API, 2 library, 3 file,
4 crawling, 5 manual), ends in the same record table via finalize().
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 목차] 이 셀은 고치지 않고 실행만 해요. 아래는 어떤 함수가 어디에 쓰이는지예요.

  공통 출구 (모든 단계가 마지막에 거치는 곳)
    collect_table()     받아온 표 → 공통 표 15칸. 내 소스 함수의 마지막 줄은 항상 이것
    finalize()          꼬리표 붙이기·숫자/날짜 정리·형식 통과/확인 필요 판정 (collect_table이 불러 씀)
    long_from_table()   칸마다 항목이 하나씩인 넓은 표 → 숫자 하나 = 한 줄 (collect_table이 불러 씀)
    check_records()     공통 표 모양이 맞는지 검사하고 요약 출력
    save_records()      CSV 두 개로 저장 (전부 / 확인 필요만)
    run_step()          예시 하나를 실행하고 ✅ ⏭ ❌ 요약표에 한 줄 남김 (실패해도 멈추지 않음)
    step_summary()      요약표 보여주기

  1단계 공식 API      read_json_table() · collect_dart() · collect_ecos() · get_key()
  2단계 라이브러리     (도구 없음: 라이브러리가 준 표를 collect_table에 바로)
  3단계 파일          read_table()
  4단계 크롤링        read_html_table() · robots_allowed() · find_col()
  5단계 수동 입력      (pd.DataFrame으로 직접) · mount_drive() · read_source_cards() · pick_sheet_title() · collect_from_cards()

  정리 도구           to_number() 숫자 바꾸기 · to_date() 날짜 바꾸기 · is_blank() 빈칸 판단
  약속된 값           NOT_APPLICABLE = "해당 없음" · SAME_AS_DATE = 공시일을 기준일과 같게
  멈춤 신호           SkipStep = 준비 부족이라 건너뜀(⏭) · CollectError = 실패(❌), 이유를 쉬운 말로
"""

import io
import json
import math
import numbers
import os
import re
import urllib.robotparser
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

COLUMNS = [
    "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각", "상태", "비고",
]
REQUIRED = [
    "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각",
]
NOT_APPLICABLE = "해당 없음"
NA_ALLOWED = {"종목코드", "종목명", "기준", "공시일"}
RELIABILITY = ("상", "중", "하")
LEVELS = {1: "공식 API", 2: "파이썬 라이브러리", 3: "파일 다운로드", 4: "크롤링", 5: "수동 입력"}
PASSED, REVIEW = "형식 통과", "확인 필요"
SAME_AS_DATE = "same_as_date"
CHECKED_MARKS = {"o", "y", "yes", "✅", "확인"}
NULL_TOKENS = {"", "-", ".", "\u2014", "n/a", "na", "nan", "none", "null"}
KST = timezone(timedelta(hours=9))
USER_AGENT = "KFA-AI-agent-study/0.2 (+https://github.com/phm0305/KFA-ai-agent-study)"


# 실패(❌). 메시지가 조원 화면에 그대로 나가서 키·전체 주소를 넣지 않아요
class CollectError(Exception):
    """Shown to members verbatim, so it must never contain keys or full URLs."""


# 준비가 안 돼서 건너뜀(⏭). 키 없음, 설정 꺼짐, 사이트가 막음 등
class SkipStep(Exception):
    pass


# 지금 한국 시각 (수집 시각 칸에 씀)
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M")


# 빈칸인지 판단 (None, 빈 글자, NaN 모두 빈칸)
def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# '1,234' '(500)' '△500' '12.5%' → 숫자. 못 바꾸면 (None, 이유)
def to_number(value):
    if is_blank(value):
        return None, None
    if isinstance(value, bool):
        return None, f"숫자로 못 바꿈: '{value}'"
    if isinstance(value, numbers.Integral):
        return int(value), None
    if isinstance(value, numbers.Real):
        return (float(value), None) if math.isfinite(value) else (None, None)
    text = str(value).strip()
    if text.lower() in NULL_TOKENS:
        return None, None
    cleaned = text.replace(",", "").replace(" ", "").rstrip("%")
    negative = False
    if cleaned.startswith("(") and cleaned.endswith(")"):
        negative, cleaned = True, cleaned[1:-1]
    elif cleaned.startswith("△"):
        negative, cleaned = True, cleaned[1:]
    if re.fullmatch(r"[+-]?\d+", cleaned):
        number = int(cleaned)
    else:
        try:
            number = float(cleaned)
        except ValueError:
            return None, f"숫자로 못 바꿈: '{text}'"
        if not math.isfinite(number):
            return None, f"숫자로 못 바꿈: '{text}'"
    return (-number if negative else number), None


_DAY = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*[-./월]\s*(\d{1,2})\s*일?\.?")
_COMPACT_DAY = re.compile(r"(\d{4})(\d{2})(\d{2})")
_MONTH = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*월?\.?")
_COMPACT_MONTH = re.compile(r"(\d{4})(\d{2})")
_QUARTER = re.compile(r"(\d{4})\s*-?\s*Q([1-4])", re.IGNORECASE)
_YEAR = re.compile(r"(\d{4})\s*년?")
_TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")


# '2026.6.30.' '20260630' '2026년 6월' '2026Q2' 등 → 표준 날짜 글자. 못 바꾸면 (None, 이유)
def to_date(value):
    """Normalize to YYYY-MM-DD, YYYY-MM, YYYYQn or YYYY; anything else is a problem."""
    if is_blank(value):
        return None, None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.strftime("%Y-%m-%d"), None
    text = _TIME_SUFFIX.sub("", str(value).strip())
    for pattern in (_DAY, _COMPACT_DAY):
        match = pattern.fullmatch(text)
        if match:
            try:
                return date(*map(int, match.groups())).isoformat(), None
            except ValueError:
                return None, f"없는 날짜: '{value}'"
    for pattern in (_MONTH, _COMPACT_MONTH):
        match = pattern.fullmatch(text)
        if match:
            year, month = map(int, match.groups())
            if 1 <= month <= 12:
                return f"{year:04d}-{month:02d}", None
            return None, f"없는 날짜: '{value}'"
    match = _QUARTER.fullmatch(text)
    if match:
        return f"{match.group(1)}Q{match.group(2)}", None
    match = _YEAR.fullmatch(text)
    if match:
        return match.group(1), None
    return None, f"날짜 모양을 모름: '{value}'"


# 이름에 그 글자가 들어간 첫 칸 찾기 (크롤링 표처럼 칸 이름이 조금씩 바뀔 때)
def find_col(table, keyword):
    for col in table.columns:
        if keyword in str(col):
            return col
    raise CollectError(f"'{keyword}'가 들어간 칸이 없어요. 지금 표의 칸: {[str(c) for c in table.columns]}")


# 넓은 표(칸마다 항목) → 긴 표(숫자 하나 = 한 줄). items·date_col을 여기서 씀
def long_from_table(table, items, date_col=None, keep=None):
    """Wide table (one column per item) -> one row per number.

    items: {raw column: (항목 이름, 단위)}. keep: {raw column: our column} copied per row.
    """
    if date_col is None:
        if isinstance(table.index, pd.RangeIndex):
            raise CollectError("날짜가 들어 있는 칸 이름을 date_col로 알려 주세요")
        table = table.reset_index()
        date_col = table.columns[0]
    keep = keep or {}
    missing = [c for c in [*items, date_col, *keep] if c not in table.columns]
    if missing:
        raise CollectError(f"표에 없는 칸: {missing}. 지금 표의 칸: {[str(c) for c in table.columns]}")
    parts = []
    for raw_col, (label, unit) in items.items():
        part = pd.DataFrame({
            "항목": label,
            "값": pd.Series(table[raw_col].tolist(), dtype=object),
            "단위": unit,
            "기준일": pd.Series(table[date_col].tolist(), dtype=object),
        })
        for raw, ours in keep.items():
            part[ours] = table[raw].tolist()
        parts.append(part)
    if not parts:
        raise CollectError("items가 비어 있어요. 어떤 칸을 숫자로 가져올지 적어 주세요")
    return pd.concat(parts, ignore_index=True)


# 꼬리표 채우기 + 숫자·날짜 정리 + 형식 통과/확인 필요 판정 (모든 단계 공통)
def finalize(table, *, level, source=None, source_url=None, reliability=None, basis=None,
             disclosed=None, base_date=None, stock_code=None, stock_name=None, unit=None, note=None):
    """Turn any table with at least 항목/값 into the shared record table.

    A cell already in the table wins over the keyword default. disclosed may be a
    date, NOT_APPLICABLE, or SAME_AS_DATE (copy 기준일, for prices and rates).
    """
    if level not in LEVELS:
        raise ValueError(f"level must be one of {list(LEVELS)}")
    defaults = {
        "종목코드": stock_code, "종목명": stock_name, "단위": unit, "기준": basis,
        "기준일": base_date, "출처": source, "출처 URL": source_url, "신뢰도": reliability,
    }
    collected_at = now_kst()
    method = f"{level} {LEVELS[level]}"
    out = []
    for raw in table.to_dict("records"):
        row = {col: raw.get(col) for col in COLUMNS}
        for col, default in defaults.items():
            if is_blank(row[col]) and default is not None:
                row[col] = default
        notes = [n for n in (raw.get("비고"), note) if not is_blank(n)]
        reasons = []

        if is_blank(row["공시일"]):
            if disclosed == SAME_AS_DATE:
                row["공시일"] = row["기준일"]
                notes.append("공시일은 기준일과 같다고 봄")
            elif disclosed is not None:
                row["공시일"] = disclosed
        row["수집 방법"] = method
        if is_blank(row["수집 시각"]):
            row["수집 시각"] = collected_at

        value, value_problem = to_number(row["값"])
        if value_problem:
            reasons.append(value_problem)
        else:
            row["값"] = value

        for col in ("기준일", "공시일"):
            if is_blank(row[col]) or row[col] == NOT_APPLICABLE:
                continue
            normalized, problem = to_date(row[col])
            if problem:
                reasons.append(f"{col} {problem}")
            else:
                row[col] = normalized

        for col in REQUIRED:
            if is_blank(row[col]) and not (col == "값" and value_problem):
                reasons.append(f"{col} 비어 있음")
        for col in COLUMNS:
            if row[col] == NOT_APPLICABLE and col not in NA_ALLOWED:
                reasons.append(f"{col}에는 '{NOT_APPLICABLE}'을 쓸 수 없음")

        base, disc = str(row["기준일"] or ""), str(row["공시일"] or "")
        if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", base) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", disc):
            if disc[:len(base)] < base:
                reasons.append("공시일이 기준일보다 빠름")
        if not is_blank(row["신뢰도"]) and row["신뢰도"] not in RELIABILITY:
            reasons.append("신뢰도는 상/중/하 중 하나")

        checked = str(raw.get("확인") or "").strip().lower() in CHECKED_MARKS
        if level == 4 and not checked:
            reasons.append("크롤링 값: 원문 화면과 한 번 대조")
        if level == 5 and not checked:
            reasons.append("손으로 옮긴 값: 원문과 대조")

        row["상태"] = REVIEW if reasons else PASSED
        row["비고"] = "; ".join([*dict.fromkeys(notes), *(f"⚠ {r}" for r in reasons)])
        out.append(row)

    records = pd.DataFrame(out, columns=COLUMNS)
    records["값"] = pd.Series([r["값"] for r in out], dtype=object)
    return records


# 모든 단계의 공통 출구: 받아온 표 → 공통 표 15칸
def collect_table(table, items=None, date_col=None, keep=None, **meta):
    """Single exit for every level: raw table in, record table out.

    Without items, the table must already have 항목/값 columns (typical for manual CSVs).
    """
    if not isinstance(table, pd.DataFrame):
        raise CollectError(f"표(DataFrame)가 아니라 {type(table).__name__}가 들어왔어요")
    if items is not None:
        table = long_from_table(table, items, date_col, keep)
    elif not {"항목", "값"} <= set(table.columns):
        raise CollectError(f"items 없이 쓰려면 '항목'과 '값' 칸이 있어야 해요. 지금 칸: {[str(c) for c in table.columns]}")
    return finalize(table, **meta)


# 확인 필요 줄의 ⚠ 이유를 종류별로 세기
def review_reasons(records):
    reasons = Counter()
    for note in records.loc[records["상태"] == REVIEW, "비고"].fillna(""):
        for part in note.split("; "):
            if part.startswith("⚠ "):
                reasons[re.sub(r"'.*'", "'…'", part[2:])] += 1
    return reasons


# 공통 표 모양 검사. 맞으면 '✅ 칸 모양 OK' 요약을 출력
def check_records(records, show=True):
    problems = []
    if not isinstance(records, pd.DataFrame):
        problems.append(f"표(DataFrame)가 아니에요: {type(records).__name__}")
    elif list(records.columns) != COLUMNS:
        missing = [c for c in COLUMNS if c not in records.columns]
        extra = [c for c in records.columns if c not in COLUMNS]
        problems.append(f"칸 이름·순서가 달라요 (빠진 칸 {missing}, 남는 칸 {extra})")
    else:
        bad_state = ~records["상태"].isin([PASSED, REVIEW])
        if bad_state.any():
            problems.append(f"상태 칸에 '{PASSED}'/'{REVIEW}' 말고 다른 값이 {int(bad_state.sum())}줄")
        blank = records[REQUIRED].apply(lambda col: col.map(is_blank)).any(axis=1)
        hidden = blank & (records["상태"] == PASSED)
        if hidden.any():
            problems.append(f"필수 칸이 비었는데 '{PASSED}'인 줄이 {int(hidden.sum())}개 (finalize를 거치지 않은 듯)")
    if show:
        if problems:
            for p in problems:
                print("❌", p)
        else:
            passed = int((records["상태"] == PASSED).sum())
            print(f"✅ 칸 모양 OK: {len(records)}줄 (형식 통과 {passed} · 확인 필요 {len(records) - passed})")
            for reason, count in review_reasons(records).most_common(5):
                print(f"   ⚠ {reason}: {count}줄")
    return not problems


# CSV 저장: 전부(stem.csv) + 확인 필요만(stem_확인필요.csv). 엑셀에서 안 깨지게 utf-8-sig
def save_records(records, stem="records"):
    all_path, review_path = f"{stem}.csv", f"{stem}_확인필요.csv"
    records.to_csv(all_path, index=False, encoding="utf-8-sig")
    records[records["상태"] == REVIEW].to_csv(review_path, index=False, encoding="utf-8-sig")
    return all_path, review_path


STEP_RESULTS = []


# 예시 하나 실행 → 요약표에 ✅ ⏭ ❌ 한 줄. 어떤 오류가 나도 '모두 실행'을 멈추지 않음
def run_step(label, level, func):
    """Run one source without ever stopping 'Run all'; errors become a summary row."""
    entry = {"단계": f"{level} {LEVELS[level]}", "예시": label, "결과": "❌", "줄 수": 0, "확인 필요": 0, "메모": ""}
    STEP_RESULTS[:] = [e for e in STEP_RESULTS if e["예시"] != label]
    STEP_RESULTS.append(entry)
    try:
        records = func()
    except SkipStep as e:
        entry.update(결과="⏭", 메모=str(e))
        print(f"⏭ {label}: {e}")
        return None
    except CollectError as e:
        entry["메모"] = str(e)
        print(f"❌ {label}: {e}")
        return None
    except requests.RequestException as e:
        entry["메모"] = f"인터넷 연결 문제로 못 받아왔어요 ({type(e).__name__}). 잠시 뒤 이 셀만 다시 실행해 주세요"
        print(f"❌ {label}: {entry['메모']}")
        return None
    except Exception as e:
        entry["메모"] = f"예상 못 한 오류 ({type(e).__name__}). 셀을 다시 실행해 보고, 그래도 안 되면 조 톡방에 공유"
        print(f"❌ {label}: {entry['메모']}")
        return None
    if not check_records(records, show=False):
        entry["메모"] = "공통 표 모양이 아니에요. check_records(결과)로 무엇이 다른지 확인"
        check_records(records)
        return None
    review = int((records["상태"] == REVIEW).sum())
    entry.update(결과="✅", **{"줄 수": len(records), "확인 필요": review})
    print(f"✅ {label}: {len(records)}줄 (확인 필요 {review}줄)")
    return records


# 지금까지 실행한 예시의 요약표
def step_summary():
    return pd.DataFrame(STEP_RESULTS, columns=["단계", "예시", "결과", "줄 수", "확인 필요", "메모"])


# 주소에 요청 보내기. 오류 메시지에는 사이트 이름만 넣어 키가 화면에 안 나오게 함
def http_get(url, params=None, timeout=30):
    host = urlparse(url).netloc
    try:
        resp = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": USER_AGENT})
    except requests.Timeout:
        raise CollectError(f"{host} 응답이 {timeout}초 넘게 없어요. 잠시 뒤 다시 실행해 주세요") from None
    except requests.RequestException:
        raise CollectError(f"{host}에 연결하지 못했어요. 인터넷이나 사이트 상태를 확인해 주세요") from None
    if resp.status_code != 200:
        hint = {401: "키나 로그인이 필요해요", 403: "사이트가 접근을 막았어요", 404: "주소가 바뀌었을 수 있어요",
                429: "요청이 너무 잦아요. 1분 뒤 다시"}.get(resp.status_code, "")
        raise CollectError(f"{host}가 {resp.status_code} 응답을 줬어요. {hint}".strip())
    return resp


# [1단계] JSON 응답에서 'a.b.c' 위치의 목록을 표로. 위치가 틀리면 그 자리에 뭐가 있는지 알려줌
def read_json_table(url, rows_path, params=None):
    """Fetch JSON and return the list found at a dotted path (e.g. 'StatisticSearch.row') as a table."""
    try:
        data = http_get(url, params).json()
    except ValueError:
        raise CollectError("응답이 JSON이 아니에요. 주소의 형식(json/xml) 부분을 확인해 주세요") from None
    current = data
    for key in rows_path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
            continue
        here = list(current) if isinstance(current, dict) else type(current).__name__
        preview = json.dumps(data, ensure_ascii=False)[:200]
        raise CollectError(f"응답에서 '{key}'를 못 찾았어요. 그 자리에 있는 것: {here} / 응답 앞부분: {preview}")
    rows = [current] if isinstance(current, dict) else current
    if not isinstance(rows, list) or not rows:
        raise CollectError(f"'{rows_path}' 아래가 비어 있어요")
    return pd.json_normalize(rows)


DART_URL = "https://opendart.fss.or.kr/api"
DART_REPORT = {1: "11013", 2: "11012", 3: "11014", 4: "11011"}
QUARTER_END = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
DART_ERRORS = {
    "010": "등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요",
    "011": "사용할 수 없는 키예요",
    "013": "조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요",
    "020": "오늘 요청 한도를 넘었어요. 내일 다시 해 주세요",
    "100": "요청 값이 잘못됐어요 (연도·분기·연결/별도)",
    "800": "DART 시스템 점검 중이에요",
}
DART_ITEMS = {
    "ifrs-full_Revenue": ("매출액", True),
    "dart_OperatingIncomeLoss": ("영업이익", True),
    "ifrs-full_ProfitLoss": ("당기순이익", True),
    "ifrs-full_Assets": ("자산총계", False),
    "ifrs-full_Liabilities": ("부채총계", False),
    "ifrs-full_Equity": ("자본총계", False),
}
_DART_CORPS = {}


def _dart_status(data):
    status = data.get("status")
    if status != "000":
        raise CollectError(f"DART {status}: {DART_ERRORS.get(status, data.get('message', '알 수 없는 응답'))}")


# [1단계] 종목코드 → DART 회사 번호 (번호표는 한 번만 받아서 기억)
def dart_corp(stock_code, api_key):
    if not _DART_CORPS:
        resp = http_get(f"{DART_URL}/corpCode.xml", {"crtfc_key": api_key})
        try:
            xml_bytes = zipfile.ZipFile(io.BytesIO(resp.content)).read("CORPCODE.xml")
        except zipfile.BadZipFile:
            try:
                _dart_status(resp.json())
            except ValueError:
                pass
            raise CollectError("DART 회사 번호표를 받지 못했어요. 키를 확인해 주세요") from None
        for item in ET.fromstring(xml_bytes).iter("list"):
            code = (item.findtext("stock_code") or "").strip()
            if code:
                _DART_CORPS[code] = (item.findtext("corp_code"), item.findtext("corp_name"))
    if stock_code not in _DART_CORPS:
        raise CollectError(f"{stock_code}: 상장사 목록에 없어요. 종목코드 6자리를 확인해 주세요")
    return _DART_CORPS[stock_code]


# [1단계] DART 재무제표 응답에서 6개 계정만 골라 3개월·누적을 나눠 적기
def dart_rows_to_long(rows, year, quarter):
    """Pick six accounts by standard account_id; flows get 3-month and cumulative rows."""
    base_note = "기준일은 12월 결산 가정"
    out, seen = [], set()
    for row in rows:
        account = row.get("account_id")
        if account not in DART_ITEMS or account in seen:
            continue
        name, is_flow = DART_ITEMS[account]
        if is_flow and row.get("sj_div") not in ("IS", "CIS"):
            continue
        seen.add(account)
        rcept_no = row.get("rcept_no", "")
        common = {
            "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
            "공시일": rcept_no[:8], "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        }
        if not is_flow:
            parts = [(name, row.get("thstrm_amount"), base_note)]
        elif quarter == 4:
            parts = [(f"{name}(연간)", row.get("thstrm_amount"), base_note + "; 4분기 3개월치 = 연간 - 3분기 누적")]
        elif quarter == 1:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note + "; 1분기는 3개월 = 누적")]
        else:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note),
                     (f"{name}(누적)", row.get("thstrm_add_amount"), base_note)]
        for label, amount, memo in parts:
            out.append({**common, "항목": label, "값": amount, "비고": memo})
    for account, (name, _) in DART_ITEMS.items():
        if account not in seen:
            out.append({"항목": name, "값": None, "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
                        "비고": f"이 회사 공시에서 표준 계정({account})을 못 찾음"})
    return pd.DataFrame(out)


# [1단계] DART 재무제표 → 공통 표 (매출액·영업이익·당기순이익·자산·부채·자본)
def collect_dart(stock_code, year, quarter, fs_div, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY 등록 후 다시)")
    if quarter not in DART_REPORT or fs_div not in ("CFS", "OFS"):
        raise CollectError("QUARTER는 1~4, FS_DIV는 'CFS'(연결) 또는 'OFS'(별도)")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[quarter], "fs_div": fs_div,
    }).json()
    _dart_status(data)
    return collect_table(
        dart_rows_to_long(data["list"], year, quarter),
        level=1, source="DART 단일회사 전체 재무제표 API", reliability="상",
        basis="연결" if fs_div == "CFS" else "별도", stock_code=stock_code, stock_name=corp_name,
        source_url="https://opendart.fss.or.kr",
    )


# [1단계] 한국은행 ECOS 통계 1개 → 공통 표. 키가 없으면 ⏭
def collect_ecos(api_key, stat_code, cycle, start, end, item_code, label, unit):
    if not api_key:
        raise SkipStep("ECOS 키가 없어요 (선택 사항: ecos.bok.or.kr에서 무료 발급 후 Secrets에 ECOS_API_KEY)")
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/1000/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    try:
        data = http_get(url).json()
    except ValueError:
        raise CollectError("ECOS 응답이 JSON이 아니에요") from None
    if "RESULT" in data:
        raise CollectError(f"ECOS: {data['RESULT'].get('MESSAGE', data['RESULT'])}")
    table = pd.json_normalize(data["StatisticSearch"]["row"])
    return collect_table(
        table, {"DATA_VALUE": (label, unit)}, date_col="TIME",
        level=1, source="한국은행 ECOS Open API", reliability="상", basis=NOT_APPLICABLE,
        source_url=f"https://ecos.bok.or.kr/#/SearchStat?statCode={stat_code}",
    )


# [3·5단계] 주소나 파일 경로의 CSV·엑셀 읽기 (한글 깨짐 cp949, 이름만 .xls인 파일도 처리)
# 칸 이름과 글자 값의 앞뒤 공백은 지워요 (화면에서 안 보이는 차이라서)
def read_table(source, sheet_name=0, skiprows=None):
    """CSV/Excel from a URL or a path. Handles cp949 CSVs and HTML disguised as .xls.

    Leading/trailing spaces in column names and text cells are removed: they are
    invisible on screen, so no reader can mean anything by them.
    """
    table = _read_table_raw(source, sheet_name, skiprows)
    table.columns = [c.strip() if isinstance(c, str) else c for c in table.columns]
    for col in table.columns:
        if not pd.api.types.is_numeric_dtype(table[col]):
            table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    return table


def _read_table_raw(source, sheet_name, skiprows):
    if str(source).startswith(("http://", "https://")):
        content, name = http_get(str(source)).content, urlparse(str(source)).path.lower()
    else:
        path = Path(source)
        if not path.exists():
            raise CollectError(f"파일이 없어요: {path.name}. 경로와 파일 이름(띄어쓰기·확장자)을 확인해 주세요")
        content, name = path.read_bytes(), path.name.lower()
    if content.lstrip()[:1] == b"<":
        try:
            return pd.read_html(io.StringIO(content.decode("utf-8", errors="replace")))[0]
        except ValueError:
            raise CollectError("파일 안이 표가 아닌 웹페이지예요. 받은 파일을 열어 확인해 주세요") from None
    if name.endswith((".xlsx", ".xlsm", ".xls")):
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=sheet_name, skiprows=skiprows)
        except ImportError:
            raise CollectError("이 엑셀 형식을 읽는 도구가 없어요. 엑셀에서 .xlsx로 다시 저장해 주세요") from None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=skiprows)
        except UnicodeDecodeError:
            continue
    raise CollectError("CSV 글자를 못 읽었어요 (utf-8, cp949 둘 다 실패). 엑셀에서 'CSV UTF-8'로 다시 저장해 주세요")


# [4단계] 사이트의 robots.txt가 코드 수집을 허락하는지 (모르면 None)
def robots_allowed(url):
    """True/False from robots.txt; None when it cannot be read (caller decides)."""
    parts = urlparse(url)
    try:
        resp = requests.get(f"{parts.scheme}://{parts.netloc}/robots.txt", timeout=15,
                            headers={"User-Agent": USER_AGENT})
    except requests.RequestException:
        return None
    if resp.status_code in (401, 403):
        return False
    if resp.status_code >= 400:
        return True
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(resp.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


# [4단계] 웹페이지 화면의 표 읽기. robots.txt가 막으면 ⏭
def read_html_table(url, match=".+", index=0):
    host = urlparse(url).netloc
    allowed = robots_allowed(url)
    if allowed is False:
        raise SkipStep(f"{host}의 robots.txt가 코드 수집을 막고 있어요. 이 자료는 5단계(수동)로 옮겨 주세요")
    if allowed is None:
        print(f"⚠ {host}의 robots.txt를 확인하지 못했어요. 사이트 이용약관을 직접 확인해 주세요")
    resp = http_get(url)
    if resp.encoding is None or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    try:
        tables = pd.read_html(io.StringIO(resp.text), match=match)
    except ValueError:
        raise CollectError(f"페이지에서 '{match}'가 들어간 표를 못 찾았어요. 표가 자바스크립트로 그려지는 페이지일 수 있어요") from None
    if index >= len(tables):
        raise CollectError(f"표가 {len(tables)}개뿐이에요. index를 0~{len(tables) - 1} 중에서 골라 주세요")
    table = tables[index]
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [" ".join(dict.fromkeys(str(c) for c in col)) for col in table.columns]
    return table


DRIVE_ROOT = "/content/drive/MyDrive/KFA_자료"
CARD_COLUMNS = ["파일명", "출처", "URL", "기준일", "신뢰도"]


# [5단계] 구글 드라이브 연결 후 KFA_자료 폴더 위치 돌려주기 (Colab 전용)
def mount_drive():
    try:
        from google.colab import drive
    except ImportError:
        raise CollectError("구글 드라이브 연결은 Colab에서만 돼요") from None
    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise CollectError("내 드라이브 맨 위에 'KFA_자료' 폴더가 없어요 (이름의 밑줄까지 똑같이)")
    return Path(DRIVE_ROOT)


# [5단계] 출처카드 읽기: CSV 경로 또는 구글 시트 제목
def read_source_cards(source="출처카드"):
    """A CSV path, or (in Colab) the title of a Google Sheet."""
    if str(source).lower().endswith(".csv") or Path(str(source)).exists():
        return read_table(source)
    try:
        from google.colab import auth
    except ImportError:
        raise CollectError("구글 시트는 Colab에서만 읽을 수 있어요. 여기서는 CSV 경로를 주세요") from None
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    client = gspread.authorize(creds)
    try:
        sheet = client.open(source).sheet1
    except gspread.SpreadsheetNotFound:
        files = client.list_spreadsheet_files()
        title = pick_sheet_title(source, [f["name"] for f in files])
        print(f"'{title}'를 {source}로 읽었어요")
        sheet = client.open_by_key(next(f["id"] for f in files if f["name"] == title)).sheet1
    return pd.DataFrame(sheet.get_all_records())


# [5단계] 시트 이름이 정확히 같지 않을 때 띄어쓰기만 다른 시트 찾기
def pick_sheet_title(wanted, titles):
    """The one title equal to wanted once all whitespace is removed; stops on zero or several."""
    squeeze = lambda s: re.sub(r"\s+", "", str(s))
    matches = [t for t in titles if squeeze(t) == squeeze(wanted)]
    if not matches:
        raise CollectError(f"'{wanted}'라는 구글 시트를 못 찾았어요. 시트 이름을 확인해 주세요")
    if len(matches) > 1:
        raise CollectError(f"'{wanted}'와 띄어쓰기만 다른 시트가 {len(matches)}개예요: {matches}. "
                           f"쓸 시트 하나만 이름을 '{wanted}'로 바꿔 주세요")
    return matches[0]


# [3·5단계] 출처카드에 적힌 파일만 읽어서 공통 표로. 못 읽는 파일은 이유와 함께 건너뜀
def collect_from_cards(cards, files_dir):
    """Read files listed on the source cards. Only CSV/XLSX already shaped as 항목/값/단위/기준일.

    An optional 공시일 card column fills 공시일 the same way 기준일 does: a cell in the file wins.
    """
    missing = [c for c in CARD_COLUMNS if c not in cards.columns]
    if missing:
        raise CollectError(f"출처카드에 없는 칸: {missing}")
    results, skipped = [], []
    for card in cards.to_dict("records"):
        name = str(card["파일명"]).strip()
        path = Path(files_dir) / name
        if not path.exists():
            skipped.append((name, "files/ 폴더에 이 이름의 파일이 없어요"))
            continue
        if path.suffix.lower() not in (".csv", ".xlsx"):
            skipped.append((name, "PDF·글 자료는 4회차에 AI가 읽어요"))
            continue
        table = read_table(path)
        if not {"항목", "값"} <= set(table.columns):
            skipped.append((name, "'항목·값·단위·기준일' 칸 모양이 아니에요. 3단계 예시처럼 items로 바꿔 주세요. "
                                  f"지금 칸: {[str(col) for col in table.columns]}"))
            continue
        level_match = re.match(r"\s*([1-5])", str(card.get("수집 방법") or ""))
        disclosed = None if is_blank(card.get("공시일")) else card["공시일"]
        results.append(collect_table(
            table, level=int(level_match.group(1)) if level_match else 5, source=card["출처"], source_url=card["URL"],
            reliability=card["신뢰도"], base_date=card["기준일"], disclosed=disclosed, note=f"출처카드: {name}",
        ))
    for name, why in skipped:
        print(f"   ⏭ {name}: {why}")
    if not results:
        raise SkipStep(f"출처카드 {len(cards)}줄 중 읽을 수 있는 숫자 파일이 없어요")
    return pd.concat(results, ignore_index=True)


# 키 읽기: Colab이면 Secrets, 내 컴퓨터면 .env. 값은 절대 출력하지 않음
def get_key(name):
    """Colab Secrets first, then .env / environment. Never print the value."""
    try:
        from google.colab import userdata
    except ImportError:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        return os.getenv(name) or None
    try:
        return userdata.get(name) or None
    except Exception:
        return None

STEP_RESULTS.clear()
print("✅ 공통 도구 준비 완료")

✅ 공통 도구 준비 완료


In [46]:
#@title ⚙️ 엔진 도구 (고치지 말고 실행만) { display-mode: "form" }
"""KFA AI agent study: report engine skeleton (3rd session).

Adds row ids and a numbered source list on top of the shared record table from
collect.py. Engine columns go on copies only; the 15 shared columns never change.
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 안내] 이 셀은 고치지 않고 실행만 해요.
    with_ids()       공통 표의 줄마다 이름표(rid)를 붙여요. 같은 사실이 두 줄이면 멈춰요
    build_sources()  출처 목록 [S1] [S2] ...을 만들고, 줄마다 어느 출처인지(source_id) 붙여요
"""

import pandas as pd


RID_PARTS = ["종목코드", "항목", "기준", "기준일", "출처"]
SOURCE_KEY = ["출처", "출처 URL"]
SOURCE_COLUMNS = ["source_id", "출처", "출처 URL", "수집 방법", "신뢰도", "기준일 범위", "줄 수"]


def _text(value):
    return "" if is_blank(value) else str(value)


def with_ids(records):
    """Copy with rid = 종목코드|항목|기준|기준일|출처; a repeated rid is the same fact twice and stops."""
    out = records.copy()
    out["rid"] = ["|".join(_text(v) for v in row) for row in out[RID_PARTS].itertuples(index=False)]
    repeated = out.loc[out["rid"].duplicated(keep=False), "rid"].unique().tolist()
    if repeated:
        raise CollectError(
            f"같은 사실(종목코드|항목|기준|기준일|출처)이 두 줄 이상 있어요: {repeated[:5]}. "
            "같은 자료를 두 번 붙였는지 확인해 주세요")
    return out


def build_sources(records):
    """Return (sources, records copy with source_id).

    One source = one (출처, 출처 URL) pair, so each DART filing keeps its own URL.
    Numbers follow the sorted pairs, not row order, so reordering cells keeps every S#.
    """
    out = records.copy()
    keys = [tuple(_text(v) for v in row) for row in out[SOURCE_KEY].itertuples(index=False)]
    ids = {key: f"S{n}" for n, key in enumerate(sorted(set(keys)), 1)}
    out["source_id"] = [ids[key] for key in keys]
    rows = []
    for (name, url), sid in ids.items():
        part = out[out["source_id"] == sid]
        row = {"source_id": sid, "출처": name, "출처 URL": url, "줄 수": len(part)}
        for col in ("수집 방법", "신뢰도"):
            values = part[col].map(_text).unique().tolist()
            if len(values) > 1:
                raise CollectError(f"[{sid}] {name}: 한 출처인데 {col}이(가) 여러 개예요 {values}. 하나로 맞춰 주세요")
            row[col] = values[0]
        dates = sorted(d for d in part["기준일"].map(_text) if d)
        if dates:
            row["기준일 범위"] = dates[0] if dates[0] == dates[-1] else f"{dates[0]} ~ {dates[-1]}"
        else:
            row["기준일 범위"] = ""
        rows.append(row)
    return pd.DataFrame(rows, columns=SOURCE_COLUMNS), out

print("✅ 엔진 도구 준비 완료")

✅ 엔진 도구 준비 완료


In [47]:
DART_API_KEY = get_key("DART_API_KEY")
print("DART 키:", "등록됨 ✅" if DART_API_KEY else "없음 ⏭ (Secrets에 DART_API_KEY + 노트북 액세스 켜기)")

DART 키: 등록됨 ✅


## 1. DART에서 두 해 받기

사업보고서(4분기)의 **연결** 재무제표에서 6개 계정을 `YEAR`와 그 전년도로 받아요. 해마다 따로 받아서, 한 해만 실패해도 어느 해인지 보여요.

In [48]:
dart_years = []
for year in (YEAR - 1, YEAR):
    result = run_step(f"DART {year} 사업보고서", 1, lambda: collect_dart(STOCK_CODE, year, 4, "CFS", DART_API_KEY))
    if result is not None:
        dart_years.append(result)

dart = pd.concat(dart_years, ignore_index=True) if dart_years else None
if dart is not None:
    display(dart[["종목명", "항목", "값", "단위", "기준", "기준일", "공시일", "상태"]])

✅ DART 2024 사업보고서: 6줄 (확인 필요 0줄)
✅ DART 2025 사업보고서: 6줄 (확인 필요 0줄)


,종목명,항목,값,단위,기준,기준일,공시일,상태
0,롯데케미칼,자산총계,34552330375053,원,연결,2024-12-31,2025-07-25,형식 통과
1,롯데케미칼,자본총계,19987944762878,원,연결,2024-12-31,2025-07-25,형식 통과
2,롯데케미칼,부채총계,14564385612175,원,연결,2024-12-31,2025-07-25,형식 통과
3,롯데케미칼,영업이익(연간),-894081004428,원,연결,2024-12-31,2025-07-25,형식 통과
4,롯데케미칼,당기순이익(연간),-1825555752135,원,연결,2024-12-31,2025-07-25,형식 통과
5,롯데케미칼,매출액(연간),20430375565357,원,연결,2024-12-31,2025-07-25,형식 통과
6,롯데케미칼,자산총계,31117333049679,원,연결,2025-12-31,2026-03-12,형식 통과
7,롯데케미칼,자본총계,17628428467279,원,연결,2025-12-31,2026-03-12,형식 통과
8,롯데케미칼,부채총계,13488904582400,원,연결,2025-12-31,2026-03-12,형식 통과
9,롯데케미칼,영업이익(연간),-943115729953,원,연결,2025-12-31,2026-03-12,형식 통과


## 2. 꼬리표 점검

`check_records`가 15칸의 이름·순서와 상태 칸을 검사하고 요약을 보여줘요. **형식 통과는 모양이 맞다는 뜻이지 숫자가 맞다는 뜻은 아니에요.** 확인 필요가 있으면 ⚠ 이유가 줄 수와 함께 나와요.

In [49]:
if dart is None:
    print("⏭ DART 결과가 없어서 건너뛰어요 (위 셀의 ❌·⏭ 메모를 먼저 확인)")
else:
    check_records(dart)

✅ 칸 모양 OK: 12줄 (형식 통과 12 · 확인 필요 0)


## 3. 줄 이름표(rid)와 출처 번호([S#])

- **rid** = `종목코드|항목|기준|기준일|출처`. 같은 rid가 두 줄이면 같은 사실을 두 번 넣은 거라 멈춰요.
- **[S#]** = (출처, 출처 URL) 한 쌍에 번호 하나. 그래서 같은 DART라도 2024·2025 사업보고서가 서로 다른 번호를 받아요. 번호는 정렬 순서로 매겨서 다시 돌려도 같아요.

In [50]:
if dart is None:
    print("⏭ DART 결과가 없어서 건너뛰어요")
else:
    dart_sources, dart_tagged = build_sources(with_ids(dart))
    display(dart_sources)
    display(dart_tagged[["source_id", "rid", "값"]])

,source_id,출처,출처 URL,수집 방법,신뢰도,기준일 범위,줄 수
0,S1,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31,6
1,S2,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-12-31,6


,source_id,rid,값
0,S1,011170|자산총계|연결|2024-12-31|DART 단일회사 전체 재무제표 API,34552330375053
1,S1,011170|자본총계|연결|2024-12-31|DART 단일회사 전체 재무제표 API,19987944762878
2,S1,011170|부채총계|연결|2024-12-31|DART 단일회사 전체 재무제표 API,14564385612175
3,S1,011170|영업이익(연간)|연결|2024-12-31|DART 단일회사 전체 재무제...,-894081004428
4,S1,011170|당기순이익(연간)|연결|2024-12-31|DART 단일회사 전체 재무...,-1825555752135
5,S1,011170|매출액(연간)|연결|2024-12-31|DART 단일회사 전체 재무제표...,20430375565357
6,S2,011170|자산총계|연결|2025-12-31|DART 단일회사 전체 재무제표 API,31117333049679
7,S2,011170|자본총계|연결|2025-12-31|DART 단일회사 전체 재무제표 API,17628428467279
8,S2,011170|부채총계|연결|2025-12-31|DART 단일회사 전체 재무제표 API,13488904582400
9,S2,011170|영업이익(연간)|연결|2025-12-31|DART 단일회사 전체 재무제...,-943115729953


## 4. 저장

왼쪽 📁 파일 탭에 CSV 세 개가 생겨요.

- `w03_dart.csv`: 공통 표 15칸 + `rid` + `source_id`
- `w03_dart_확인필요.csv`: 확인 필요 줄만
- `w03_dart_sources.csv`: 출처 목록 [S#]

In [51]:
def save_engine(tagged, sources, stem):
    paths = save_records(tagged, stem)
    sources.to_csv(f"{stem}_sources.csv", index=False, encoding="utf-8-sig")
    print("💾 저장:", *paths, f"{stem}_sources.csv")

if dart is None:
    print("⏭ DART 결과가 없어서 건너뛰어요")
else:
    save_engine(dart_tagged, dart_sources, "w03_dart")

💾 저장: w03_dart.csv w03_dart_확인필요.csv w03_dart_sources.csv


## 5. 수동 입구: 내 드라이브의 출처카드 읽기

2회차에 만든 `KFA_자료` 폴더와 출처카드를 그대로 써요. 설정 셀에서 `USE_DRIVE = True`로 바꾸면 드라이브 연결 창과 구글 로그인 창이 차례로 떠요(둘 다 허용).
출처카드에 **적힌 파일만**, 그중 `항목, 값, 단위, 기준일` 칸으로 정리한 CSV·엑셀만 읽어요. 건너뛴 파일은 이유가 한 줄씩 나와요.
출처카드에 `공시일` 칸을 더하면(선택) 그 파일 줄들의 공시일로 쓰여요. 파일 안에 `공시일` 칸이 있으면 파일 쪽이 먼저예요.

In [52]:
def drive_records():
    if not USE_DRIVE:
        raise SkipStep("USE_DRIVE = False (KFA_자료 폴더를 만들었으면 True로)")
    root = mount_drive()
    return collect_from_cards(read_source_cards(CARD_SHEET), root / "files")

drive = run_step("내 드라이브 출처카드", 5, drive_records)
if drive is not None:
    display(drive)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 내 드라이브 출처카드: 4줄 (확인 필요 0줄)


,종목코드,종목명,항목,값,단위,기준,기준일,공시일,출처,출처 URL,신뢰도,수집 방법,수집 시각,상태,비고
0,None,None,매출액,18483005,백만원,연결,2025-12-31,2026-03-04,DART 롯데케미칼 연결 잠정실적 공정공시,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,5 수동 입력,2026-11-08 22:25,형식 통과,출처카드: 롯데케미칼_2025잠정실적_2026-03.csv
1,None,None,영업이익,-943116,백만원,연결,2025-12-31,2026-03-04,DART 롯데케미칼 연결 잠정실적 공정공시,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,5 수동 입력,2026-11-08 22:25,형식 통과,출처카드: 롯데케미칼_2025잠정실적_2026-03.csv
2,None,None,당기순이익,-2476176,백만원,연결,2025-12-31,2026-03-04,DART 롯데케미칼 연결 잠정실적 공정공시,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,5 수동 입력,2026-11-08 22:25,형식 통과,출처카드: 롯데케미칼_2025잠정실적_2026-03.csv
3,None,None,영업이익,-914535,백만원,연결,2024-12-31,2026-03-04,DART 롯데케미칼 연결 잠정실적 공정공시,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,5 수동 입력,2026-11-08 22:25,형식 통과,출처카드: 롯데케미칼_2025잠정실적_2026-03.csv


## 6. 자동 + 수동을 합쳐서 같은 흐름 다시

DART와 출처카드를 이어 붙이고 2~4번(점검 → rid·[S#] → 저장)을 한 번에 돌려요. 저장 이름은 `w03_records`예요.
출처카드 파일은 자기 [S#]를 따로 받아요. 출처가 늘면 기존 번호가 밀릴 수 있어요. `USE_DRIVE = False`면 DART만 다시 돌아서 위와 같은 결과가 나와요.

In [53]:
def engine_pass(records, stem):
    if not check_records(records):
        return None
    try:
        sources, tagged = build_sources(with_ids(records))
    except CollectError as e:
        print("❌", e)
        return None
    display(sources)
    save_engine(tagged, sources, stem)
    return tagged

parts = [f for f in (dart, drive) if f is not None]
merged = engine_pass(pd.concat(parts, ignore_index=True), "w03_records") if parts else None

✅ 칸 모양 OK: 16줄 (형식 통과 16 · 확인 필요 0)


,source_id,출처,출처 URL,수집 방법,신뢰도,기준일 범위,줄 수
0,S1,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31,6
1,S2,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-12-31,6
2,S3,DART 롯데케미칼 연결 잠정실적 공정공시,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,5 수동 입력,상,2024-12-31 ~ 2025-12-31,4


💾 저장: w03_records.csv w03_records_확인필요.csv w03_records_sources.csv


## ✍️ 과제: 내 소스를 종목 3개 이상에 붙이기

2회차에 만든 `my_records()`를 **종목코드를 받는 모양** `my_records(code)`로 바꿔 아래에 붙여요. `MY_TARGETS`의 종목마다 한 번씩 돌아요.

1. `MY_LEVEL`에 내 소스의 수집 방법 단계(1~5), `MY_TARGETS`에 종목코드 3개 이상을 적어요
2. `raise SkipStep(...)` 줄을 지우고, 그 아래를 2회차에 성공한 코드로 바꿔요
3. 주소나 요청 값에 종목코드가 들어가는 자리에 `code`를 넣고, `collect_table(...)` 끝에 `stock_code=code`를 남겨요
4. 금리처럼 종목과 상관없는 소스면 `MY_TARGETS`에 통계 번호 3개를 넣고 `stock_code=code` 줄을 지워요

종목마다 ✅ ⏭ ❌가 한 줄씩 나와요. 실패한 종목은 지우지 말고 그대로 두세요. 아래 실패 기록 표에 이유가 남아요.

In [54]:
# ✍️ 내 소스: DART 단일회사 전체 재무제표 API (연결, 사업보고서)
# 롯데케미칼 신용평가 지표 6개 계산용 재무 수치를 3개 회사에 붙이기

MY_LEVEL = 1
MY_TARGETS = ["011170", "006650", "051910"]   # 롯데케미칼, 대한유화, LG화학
MY_YEAR = "2025"
MY_REPORT = "11011"                          # 사업보고서
DATES = ("2025-12-31", "2024-12-31")         # 당기 / 전기 (12월 결산 가정)
PREV_COLS = ["frmtrm_amount", "frmtrm_add_amount", "frmtrm_q_amount"]

ACCOUNTS = {   # 표준 계정ID: (표 종류, 항목 이름, 꼭 있어야 하나, 쓰는 지표)
    "ifrs-full_Revenue":                          ("IS", "매출액", True, "EBITDA 마진"),
    "dart_OperatingIncomeLoss":                   ("IS", "영업이익", True, "EBITDA"),
    "ifrs-full_FinanceCosts":                     ("IS", "금융원가", True, "EBITDA/금융비용"),
    "ifrs-full_AdjustmentsForDepreciationExpense": ("CF", "감가상각비", False, "EBITDA"),
    "ifrs-full_AdjustmentsForAmortisationExpense": ("CF", "무형자산상각비", False, "EBITDA"),
    "ifrs-full_CashFlowsFromUsedInOperatingActivities": ("CF", "영업활동현금흐름", True, "영업현금흐름/총차입금"),
    "ifrs-full_CashAndCashEquivalents":           ("BS", "현금및현금성자산", True, "순차입금/EBITDA"),
    "ifrs-full_ShortTermBorrowings":              ("BS", "단기차입금", False, "총차입금"),
    "ifrs-full_CurrentPortionOfLongtermBorrowings": ("BS", "유동성장기차입금", False, "총차입금"),
    "ifrs-full_LongtermBorrowings":               ("BS", "장기차입금", False, "총차입금"),
    "ifrs-full_Liabilities":                      ("BS", "부채총계", True, "부채비율"),
    "ifrs-full_Equity":                           ("BS", "자본총계", True, "부채비율"),
    "ifrs-full_Assets":                           ("BS", "자산총계", True, "차입금의존도"),
}

def my_records(code):
    key = get_key("DART_API_KEY")
    if not key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY + 노트북 액세스)")

    corp_code, corp_name = dart_corp(code, key)                      # 종목코드 → DART 회사번호
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": key, "corp_code": corp_code, "bsns_year": MY_YEAR,
        "reprt_code": MY_REPORT, "fs_div": "CFS"}).json()           # CFS = 연결
    _dart_status(data)                                               # DART 010·013·020이면 ❌로 메시지 남김
    raw = pd.json_normalize(data["list"])
    no = raw["rcept_no"].iloc[0]                                     # 회사마다 다른 접수번호 (고정 안 함)
    raw["sj"] = raw["sj_div"].replace({"CIS": "IS"})                 # 포괄손익계산서도 손익으로

    rows, skipped = [], []
    for acc, (sj, name, required, use) in ACCOUNTS.items():
        hit = raw[(raw["account_id"] == acc) & (raw["sj"] == sj)]
        if hit.empty:
            if required:
                raise CollectError(f"{corp_name} {name} 계정이 없어요 ({acc})")
            skipped.append(name)
            continue
        r = hit.iloc[0]
        prev_col = next((c for c in PREV_COLS if c in r.index and not is_blank(r[c])), None)
        rows += [{"항목": name, "값": r["thstrm_amount"], "단위": "원", "기준일": DATES[0],
                  "비고": f"{use}용"},
                 {"항목": name, "값": r[prev_col] if prev_col else None, "단위": "원", "기준일": DATES[1],
                  "비고": f"{use}용 · 사업보고서 비교 칸"}]

    if skipped:   # 함정 사례(T9 계열) 확인용: 못 찾은 계정과 비슷한 이름의 줄을 보여줌
        print(f"[{corp_name}] 표준 계정으로 못 찾아서 건너뛴 계정:", skipped)
        cand = raw[raw["account_nm"].str.contains("차입금|사채|리스부채|감가상각|상각비")
                   & raw["sj"].isin(["BS", "CF"])]
        print(cand[["sj", "account_id", "account_nm", "thstrm_amount"]].drop_duplicates().to_string())

    return collect_table(
        pd.DataFrame(rows),
        level=MY_LEVEL,
        source="DART 단일회사 전체 재무제표 API",
        source_url=f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={no}",
        reliability="상",
        basis="연결",
        disclosed=f"{no[:4]}-{no[4:6]}-{no[6:8]}",                   # 접수번호 앞 8자리 = 공시일
        stock_code=code, stock_name=corp_name)

mine_parts = []
for code in MY_TARGETS:
    result = run_step(f"내 소스 {code}", MY_LEVEL, lambda: my_records(code))
    if result is not None:
        mine_parts.append(result)

mine = engine_pass(pd.concat(mine_parts, ignore_index=True), "w03_my_source") if mine_parts else None
if mine is not None:
    display(mine[["source_id", "종목명", "항목", "값", "기준일", "공시일", "상태"]])

[롯데케미칼] 표준 계정으로 못 찾아서 건너뛴 계정: ['감가상각비', '무형자산상각비', '단기차입금', '유동성장기차입금']
     sj                                                           account_id       account_nm  thstrm_amount
36   BS                                    ifrs-full_CurrentLeaseLiabilities          유동성리스부채    54377937796
41   BS                                        ifrs-full_ShorttermBorrowings           차입금및사채  4442331827433
46   BS                                         ifrs-full_LongtermBorrowings           차입금및사채  4957055835186
47   BS                                 ifrs-full_NoncurrentLeaseLiabilities             리스부채   160690555902
97   CF                                               dart_ProceedsFromBonds           사채의 발행              0
100  CF      ifrs-full_ProceedsFromBorrowingsClassifiedAsFinancingActivities          차입금의 차입  6979209183809
109  CF                                     dart_RepaymentsOfBondWithWarrant  유동성신주인수권부사채의 상환              0
110  CF                                               da

,source_id,출처,출처 URL,수집 방법,신뢰도,기준일 범위,줄 수
0,S1,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31 ~ 2025-12-31,18
1,S2,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31 ~ 2025-12-31,18
2,S3,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31 ~ 2025-12-31,18


💾 저장: w03_my_source.csv w03_my_source_확인필요.csv w03_my_source_sources.csv


,source_id,종목명,항목,값,기준일,공시일,상태
0,S2,롯데케미칼,매출액,18483005314922,2025-12-31,2026-03-12,형식 통과
1,S2,롯데케미칼,매출액,19894809600093,2024-12-31,2026-03-12,형식 통과
2,S2,롯데케미칼,영업이익,-943115729953,2025-12-31,2026-03-12,형식 통과
3,S2,롯데케미칼,영업이익,-914535316186,2024-12-31,2026-03-12,형식 통과
4,S2,롯데케미칼,금융원가,866366263475,2025-12-31,2026-03-12,형식 통과
5,S2,롯데케미칼,금융원가,800430402601,2024-12-31,2026-03-12,형식 통과
6,S2,롯데케미칼,영업활동현금흐름,488927763234,2025-12-31,2026-03-12,형식 통과
7,S2,롯데케미칼,영업활동현금흐름,1542393674686,2024-12-31,2026-03-12,형식 통과
8,S2,롯데케미칼,현금및현금성자산,1870312010924,2025-12-31,2026-03-12,형식 통과
9,S2,롯데케미칼,현금및현금성자산,2111786094945,2024-12-31,2026-03-12,형식 통과


## API 실패 기록

API가 실패하는 흔한 이유 세 가지 중 두 가지를 일부러 일으켜서 메시지 모양을 봐요.

- **키 틀림**: 가짜 키로 요청 → DART 010
- **데이터 없음**: 아직 안 나온 `YEAR + 1` 사업보고서를 요청 → DART 013
- **호출 제한**: 하루 한도를 넘겨야 나와서 일부러 만들지 않아요. DART는 `DART 020: 오늘 요청 한도를 넘었어요. 내일 다시 해 주세요`로 알려줘요

아래 표에는 ✅가 아닌 줄만 나와요. 내 소스가 실패한 종목도 여기 같이 나오니, 메모를 보고 위 셋 중 어디에 해당하는지 가이드의 실패 기록에 적어요.

In [55]:
run_step("실패 예시: 키 틀림", 1, lambda: collect_dart(STOCK_CODE, YEAR, 4, "CFS", "not-a-real-key"))
run_step("실패 예시: 데이터 없음", 1, lambda: collect_dart(STOCK_CODE, YEAR + 1, 4, "CFS", DART_API_KEY))

summary = step_summary()
display(summary[summary["결과"] != "✅"])

❌ 실패 예시: 키 틀림: DART 010: 등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요
❌ 실패 예시: 데이터 없음: DART 013: 조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요


,단계,예시,결과,줄 수,확인 필요,메모
6,1 공식 API,실패 예시: 키 틀림,❌,0,0,DART 010: 등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요
7,1 공식 API,실패 예시: 데이터 없음,❌,0,0,DART 013: 조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요


## 📤 제출

**파일 → GitHub에 사본 저장** → 내 레포 `kfa-ai-agent` → 파일 경로 `w03_엔진.ipynb` → 커밋 메시지 `3회차 과제`.
자세한 순서와 결과 읽는 법은 [3회차 가이드](https://github.com/phm0305/KFA-ai-agent-study/blob/main/docs/sessions/03/README.md)를 봐 주세요.

## 📤 과제

## 📝 3회차 기록

### API 실패 기록

| 종목·요청 | 보인 메시지 | 종류 | 한 일 |
|---|---|---|---|
| 실패 예시: 키 틀림 | DART 010: 등록되지 않은 키예요 | 키 틀림 | 일부러 낸 예시 |
| 실패 예시: 데이터 없음 (2026 사업보고서) | DART 013: 조회된 데이터가 없어요 | 데이터 없음 | 일부러 낸 예시 (아직 공시 전) |
| 내 소스 011170·006650·051910 | 실패 없음 | - | - |

### 함정: 단기 차입금이 표준 계정으로 안 잡혀 총차입금이 작아짐 (T9 계열)

- 무엇: 표준 계정ID로만 찾으면 회사가 다른 ID로 공시한 단기 차입이 빠져요. 줄 자체가 안 생겨 확인 필요도 안 뜨고(54줄 모두 형식 통과), 총차입금이 작게 잡혀요.
- 관찰: 2025 사업보고서(연결)

| 종목 | 빠진 단기 차입 | 잡힌 장기 차입 | 누락 비율 |
|---|---|---|---|
| 롯데케미칼 (011170) | 4조 4,423억원 | 4조 9,571억원 | 47.3% |
| 대한유화 (006650) | 1,494억원 | 1,274억원 | 54.0% |
| LG화학 (051910) | 11조 7,381억원 | 22조 738억원 | 34.7% |

  - 감가상각비·무형자산상각비: 세 회사 모두 표준 계정·이름 검색으로 안 나옴
- 처리 방향: 계정ID 후보를 여러 개 두고 찾기 / 못 찾은 계정은 빈칸 줄로 남겨 확인 필요로 넘기기 / 감가상각비는 사업보고서 주석에서 수동 입구로

### 판단 질문: DART 숫자와 네이버 금융·리포트 숫자가 다르면?

둘 다 맞는데 기준이 다른 경우가 대부분이에요. 단위·기간·공시 시점을 맞춘 뒤에도 다를 때만 한쪽이 틀린 거예요.

- 비교 1 (단위): 롯데케미칼 2025 영업이익. 사업보고서 API −943,115,729,953원 vs 잠정공시(2026-03-04) −943,116백만원 → 같은 숫자, 단위만 다름
- 비교 2 (시점): 같은 2025 영업손실. 2026-02-04 실적발표 보도 9,436억원 vs 3/4 공시 9,431억원 → 감사 전·후 버전 차이
- 비교 3 (기간): 같은 3/4 공시 안에서 영업이익 당해실적(4분기 3개월) −433,474백만원 vs 누계(연간) −943,116백만원

→ 그래서 숫자마다 단위·기준·기준일·공시일 꼬리표가 있어야 비교할 수 있어요.